# Steam Games -> Silver y DLQ

Primero se tipa y deduplica el catálogo. Después se escribe Silver con las filas válidas y DLQ con las filas que incumplen reglas.

In [ ]:
import json
dbutils.widgets.text("catalog", "steam_game_intelligence")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("dlq_schema", "dlq")
dbutils.widgets.text("cutoff_us", "")
dbutils.widgets.text("run_id", "")
catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
dlq_schema = dbutils.widgets.get("dlq_schema")
cutoff_us = dbutils.widgets.get("cutoff_us")
run_id = dbutils.widgets.get("run_id")

In [ ]:
%sql
-- Silver: una fila vigente por app_id hasta el corte común.
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.' || :silver_schema || '.games') AS
WITH typed AS (
  SELECT
    try_cast(app_id AS BIGINT) AS app_id,
    name,
    lower(trim(name)) AS name_normalized,
    try_to_date(release_date) AS release_date,
    try_cast(price AS DECIMAL(18,2)) AS price,
    price_status,
    estimated_owners,
    try_cast(regexp_extract(estimated_owners, '^([0-9]+)', 1) AS DECIMAL(38,0)) AS owners_min,
    try_cast(regexp_extract(estimated_owners, '([0-9]+)[^0-9]*$', 1) AS DECIMAL(38,0)) AS owners_max,
    developers, publishers,
    from_json(genres, 'array<string>') AS genres,
    from_json(categories, 'array<string>') AS categories,
    from_json(tags, 'array<string>') AS tags,
    try_cast(positive AS BIGINT) AS positive,
    try_cast(negative AS BIGINT) AS negative,
    try_cast(recommendations AS DECIMAL(38,0)) AS recommendations,
    try_cast(peak_ccu AS BIGINT) AS peak_ccu,
    metacritic_score, user_score,
    average_playtime_forever, median_playtime_forever,
    average_playtime_2weeks, median_playtime_2weeks,
    short_description, about_the_game, detailed_description, notes,
    try_cast(achievements AS DECIMAL(38,0)) AS achievements,
    dlc_count, packages, supported_languages, full_audio_languages,
    windows, mac, linux, header_image, screenshots, movies, website,
    support_url, support_email, metacritic_url,
    steam_store_available, steam_spy_available, _ingested_at, _source
  FROM IDENTIFIER(:catalog || '.' || :bronze_schema || '.steam_games')
  WHERE _ingested_at <= timestamp_micros(try_cast(:cutoff_us AS BIGINT))
), ranked AS (
  SELECT *, row_number() OVER (PARTITION BY app_id ORDER BY _ingested_at DESC, sha2(to_json(struct(*)), 256) DESC) AS rn
  FROM typed
), candidate AS (
  SELECT *, filter(array(
    CASE WHEN app_id IS NULL OR app_id <= 0 THEN 'app_id_invalido' END,
    CASE WHEN name IS NULL OR trim(name) = '' THEN 'name_vacio' END,
    CASE WHEN price < 0 THEN 'price_negativo' END,
    CASE WHEN positive < 0 OR negative < 0 OR peak_ccu < 0 THEN 'conteo_negativo' END,
    CASE WHEN estimated_owners IS NOT NULL AND trim(estimated_owners) <> '' AND (owners_min IS NULL OR owners_max IS NULL) THEN 'estimated_owners_invalido' END
  ), x -> x IS NOT NULL) AS error_reasons
  FROM ranked WHERE rn = 1
)
SELECT * EXCEPT (rn, error_reasons)
FROM candidate WHERE size(error_reasons) = 0;

In [ ]:
%sql
-- DLQ: conserva el payload y el motivo de cada fila rechazada.
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.' || :dlq_schema || '.games_rejected') AS
WITH typed AS (
  SELECT *, try_cast(app_id AS BIGINT) AS app_id_typed,
    try_cast(price AS DECIMAL(18,2)) AS price_typed,
    try_cast(positive AS BIGINT) AS positive_typed,
    try_cast(negative AS BIGINT) AS negative_typed,
    try_cast(peak_ccu AS BIGINT) AS peak_ccu_typed,
    try_cast(regexp_extract(estimated_owners, '^([0-9]+)', 1) AS DECIMAL(38,0)) AS owners_min,
    try_cast(regexp_extract(estimated_owners, '([0-9]+)[^0-9]*$', 1) AS DECIMAL(38,0)) AS owners_max
  FROM IDENTIFIER(:catalog || '.' || :bronze_schema || '.steam_games')
  WHERE _ingested_at <= timestamp_micros(try_cast(:cutoff_us AS BIGINT))
), ranked AS (
  SELECT *, row_number() OVER (PARTITION BY app_id_typed ORDER BY _ingested_at DESC, sha2(to_json(struct(*)), 256) DESC) AS rn
  FROM typed
), rejected AS (
  SELECT *, filter(array(
    CASE WHEN app_id_typed IS NULL OR app_id_typed <= 0 THEN 'app_id_invalido' END,
    CASE WHEN name IS NULL OR trim(name) = '' THEN 'name_vacio' END,
    CASE WHEN price_typed < 0 THEN 'price_negativo' END,
    CASE WHEN positive_typed < 0 OR negative_typed < 0 OR peak_ccu_typed < 0 THEN 'conteo_negativo' END,
    CASE WHEN estimated_owners IS NOT NULL AND trim(estimated_owners) <> '' AND (owners_min IS NULL OR owners_max IS NULL) THEN 'estimated_owners_invalido' END
  ), x -> x IS NOT NULL) AS error_reasons
  FROM ranked WHERE rn = 1
)
SELECT sha2(concat_ws('|', coalesce(cast(app_id_typed AS STRING), 'NULL'), cast(_ingested_at AS STRING), to_json(struct(*))), 256) AS _reject_key,
  app_id_typed AS app_id, array_join(error_reasons, '; ') AS motivo_rechazo,
  error_reasons AS reglas_violadas, to_json(struct(*)) AS payload,
  :run_id AS _run_id, current_timestamp() AS _rejected_at
FROM rejected WHERE size(error_reasons) > 0;

In [ ]:
dbutils.notebook.exit('{"status":"completed","table":"games"}')